In [1]:
# ============================================================
# Unsloth 3-Stage Equipment (health systems) failure smart assistant tool  Fine-Tuning Pipeline

# ============================================================

# -------------------------
# 1. Install libraries
# -------------------------
!pip -q install unsloth
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2
!pip -q install -U pymupdf datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6

In [2]:
# -------------------------
# 2. Imports
# -------------------------
import os
import re
import gc
import time
import json
import unicodedata
import warnings
from typing import List, Dict, Any

warnings.filterwarnings("ignore")

import torch
import fitz  # PyMuPDF
from datasets import Dataset, load_dataset

import unsloth  # keep this import early
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig

try:
    from unsloth import PatchDPOTrainer
    PatchDPOTrainer()
    print("DPO patch applied.")
except Exception as e:
    print("DPO patch skipped:", repr(e))

from trl import DPOTrainer, DPOConfig

assert torch.cuda.is_available(), "GPU not found. In Colab: Runtime -> Change runtime type -> GPU"
print("GPU:", torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
DPO patch applied.
GPU: Tesla T4


In [3]:
# -------------------------
# 3. Real file paths
# -------------------------
non_instruction_data_path = "/content/data.pdf"


for path in [non_instruction_data_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}. Please upload this file to Colab.")


In [4]:
# -------------------------
# 4. Simple config
# -------------------------
BASE_MODEL_NAME = "unsloth/tinyllama-bnb-4bit"

MAX_SEQ_LENGTH = 512
SEED = 42
MIN_CHARS_PER_PARAGRAPH = 80

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0

BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
WARMUP_STEPS = 5
LOGGING_STEPS = 1

# Classroom demo steps. Increase for serious training.
STAGE1_MAX_STEPS = 30
STAGE2_MAX_STEPS = 30
STAGE3_MAX_STEPS = 30

STAGE1_LR = 2e-4
STAGE2_LR = 1e-4
STAGE3_LR = 5e-5

DPO_BETA = 0.1

OUTPUT_ROOT = "/content/unsloth_pharma_merge_reload_outputs"

STAGE1_ADAPTER_DIR = f"{OUTPUT_ROOT}/stage1_non_instruction_adapter"
STAGE1_MERGED_DIR  = f"{OUTPUT_ROOT}/stage1_non_instruction_merged_model"

In [5]:
# path for merged model
for path in [
    OUTPUT_ROOT,
    STAGE1_ADAPTER_DIR,
    STAGE1_MERGED_DIR,
]:
    os.makedirs(path, exist_ok=True)

## helper functions

In [6]:
# -------------------------
# 5. Helper functions
# -------------------------
def clear_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()

In [7]:
def train_and_measure(trainer, stage_name: str):
    clear_gpu_memory()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    start_time = time.time()
    result = trainer.train()
    torch.cuda.synchronize()

    train_time = round(time.time() - start_time, 2)
    peak_allocated = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
    peak_reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 3)

    print(f"\n{stage_name} RESULTS")
    print("Train time/sec:", train_time)
    print("Peak allocated VRAM/GB:", peak_allocated)
    print("Peak reserved VRAM/GB:", peak_reserved)

    return result


In [8]:

def build_instruction_prompt(instruction: str, input_text: str = "") -> str:
    instruction = str(instruction).strip()
    input_text = str(input_text).strip()

    if input_text:
        return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

    return f"### Instruction:\n{instruction}\n\n### Response:\n"

In [9]:
def generate_answer(model, tokenizer, instruction: str, input_text: str = "", max_new_tokens: int = 150):
    FastLanguageModel.for_inference(model)

    prompt = build_instruction_prompt(instruction, input_text)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_tokens = inputs["input_ids"].shape[-1]
    generated_tokens = output[0][input_tokens:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

In [10]:
def load_unsloth_model_with_lora(model_name_or_path: str):
    """
    Loads a base or merged model in 4-bit and attaches a fresh LoRA adapter.
    """

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name_or_path,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

    model.print_trainable_parameters()
    return model, tokenizer

In [11]:
def save_adapter_and_merge(model, tokenizer, adapter_dir: str, merged_dir: str, stage_name: str):
    """
    Saves LoRA adapter separately and also saves a merged standalone model.
    The merged model becomes the starting point for the next stage.
    """

    print(f"\nSaving {stage_name} adapter...")
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f"{stage_name} adapter saved to:", adapter_dir)

    print(f"\nMerging {stage_name} adapter with base model...")
    FastLanguageModel.for_training(model)

    model.save_pretrained_merged(
        merged_dir,
        tokenizer,
        save_method="merged_16bit",
    )

    print(f"{stage_name} merged model saved to:", merged_dir)

## Extracting raw text

In [12]:
def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    pages = []

    with fitz.open(pdf_path) as doc:
        for page_number, page in enumerate(doc, start=1):
            text = page.get_text("text").strip()
            if text:
                pages.append({
                    "page": page_number,
                    "text": text,
                })

    return pages


In [13]:
def clean_pdf_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # Join words broken by hyphen and newline
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Remove page-number-only lines
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Normalize spaces and paragraph breaks
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    paragraphs = []

    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()

        if paragraph:
            paragraphs.append(paragraph)

    return "\n\n".join(paragraphs)



In [14]:
def build_pdf_dataset(pdf_path: str) -> Dataset:
    pages = extract_pdf_pages(pdf_path)
    records = []

    for page in pages:
        cleaned_text = clean_pdf_text(page["text"])

        for para_id, paragraph in enumerate(cleaned_text.split("\n\n"), start=1):
            paragraph = paragraph.strip()

            if len(paragraph) >= MIN_CHARS_PER_PARAGRAPH:
                records.append({
                    "text": paragraph,
                    "source_page": page["page"],
                    "paragraph_id": para_id,
                })

    if len(records) == 0:
        raise ValueError("No usable paragraph found. Try reducing MIN_CHARS_PER_PARAGRAPH.")

    print("PDF pages extracted:", len(pages))
    print("Paragraph records:", len(records))
    print("\nSample paragraph:\n", records[0]["text"][:700])

    return Dataset.from_list(records)

In [15]:
stage1_dataset = build_pdf_dataset(non_instruction_data_path)

PDF pages extracted: 137
Paragraph records: 137

Sample paragraph:
 Medical Imaging Equipment Failure Diagnosis Knowledge Base Introduction This handbook provides domain-specific knowledge for troubleshooting CT scanners, MRI systems, ultrasound systems, digital radiography, mammography, fluoroscopy, C-arms and PET/CT systems. It explains major subsystems, operating principles, common failure modes, symptoms, likely root causes, inspection procedures and service considerations. The material is intended as a reference for biomedical engineers and field service personnel. This handbook provides domain-specific knowledge for troubleshooting CT scanners, MRI systems, ultrasound systems, digital radiography, mammography, fluoroscopy, C-arms and PET/CT systems. It


 ## Non-instruction continued pretraining

In [16]:
print("\n==============================")
print("STAGE 1: PDF RAW TEXT TRAINING")
print("==============================")

stage1_model, tokenizer = load_unsloth_model_with_lora(BASE_MODEL_NAME)

FastLanguageModel.for_training(stage1_model)

stage1_config = SFTConfig(
    output_dir=f"{OUTPUT_ROOT}/stage1_logs",

    max_steps=STAGE1_MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=STAGE1_LR,
    warmup_steps=WARMUP_STEPS,

    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=True,

    seed=SEED,
)

stage1_trainer = SFTTrainer(
    model=stage1_model,
    processing_class=tokenizer,
    train_dataset=stage1_dataset,
    args=stage1_config,
)

train_and_measure(stage1_trainer, "STAGE 1 - NON-INSTRUCTION PDF TRAINING")

save_adapter_and_merge(
    model=stage1_model,
    tokenizer=tokenizer,
    adapter_dir=STAGE1_ADAPTER_DIR,
    merged_dir=STAGE1_MERGED_DIR,
    stage_name="Stage 1",
)



STAGE 1: PDF RAW TEXT TRAINING
==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.7.2 patched 22 layers with 22 QKV layers, 22 O layers and 22 MLP layers.


trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/137 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=6):   0%|          | 0/137 [00:00<?, ? examples/s]

...

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:16<00:00, 16.96s/it]


Unsloth: Merge process complete. Saved to `/content/unsloth_pharma_merge_reload_outputs/stage1_non_instruction_merged_model`
Stage 1 merged model saved to: /content/unsloth_pharma_merge_reload_outputs/stage1_non_instruction_merged_model


## Q&A Chatbot: Base Model vs. Fine-tuned Model (Stage 1)

In [17]:
# -------------------------
# 6. Q&A Chatbot
# -------------------------
print("\n==============================")
print("LOADING BASE MODEL FOR INFERENCE")
print("==============================")

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token
base_tokenizer.padding_side = "right"

print("Base model loaded.")


LOADING BASE MODEL FOR INFERENCE
==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Base model loaded.


In [18]:
print("\n========================================")
print("LOADING FINE-TUNED MODEL (STAGE 1) FOR INFERENCE")
print("========================================")

stage1_merged_model, stage1_merged_tokenizer = FastLanguageModel.from_pretrained(
    model_name=STAGE1_MERGED_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

if stage1_merged_tokenizer.pad_token is None:
    stage1_merged_tokenizer.pad_token = stage1_merged_tokenizer.eos_token
stage1_merged_tokenizer.padding_side = "right"

print("Fine-tuned (Stage 1) model loaded.")


LOADING FINE-TUNED MODEL (STAGE 1) FOR INFERENCE
==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Fine-tuned (Stage 1) model loaded.


### Sample Questions and Answers

In [ ]:
sample_questions = [
    "My CT scanner is producing circular ring artifacts on every scan. What is the most likely cause and what should I inspect first?",
    "The CT scanner stopped scanning because of a tube overheating alarm. What components are likely responsible?",
    "How does quantization reduce model size?",
    "My MRI system displays a low helium pressure alarm during scanning. What could be causing this?",
    "The ultrasound probe is connected, but the system cannot detect it. How should I troubleshoot this issue?",
    "The CT gantry will not rotate after startup. Which components should I inspect?",
    "MRI images contain zipper artifacts. What usually causes this type of artifact?",
    "The ultrasound image is completely black even though the machine powers on. What are the possible causes?",
    "What preventive maintenance should be performed on a CT scanner to reduce unexpected failures?",
    "How can I differentiate between a detector failure and a detector calibration problem in a CT scanner?",
    "My imaging system repeatedly restarts during operation without displaying any error code. What are the most likely causes?",
]

for i, question in enumerate(sample_questions):
    print(f"\n--- Question {i+1} ---")
    print(f"Q: {question}")

    # Generate answer from base model
    base_answer = generate_answer(base_model, base_tokenizer, instruction=question)
    print(f"Base Model A: {base_answer}")

clear_gpu_memory()

In [20]:
sample_questions = [
      "My CT scanner is producing circular ring artifacts on every scan. What is the most likely cause and what should I inspect first?",
    "The CT scanner stopped scanning because of a tube overheating alarm. What components are likely responsible?",
    "How does quantization reduce model size?",
    "My MRI system displays a low helium pressure alarm during scanning. What could be causing this?",
    "The ultrasound probe is connected, but the system cannot detect it. How should I troubleshoot this issue?",
    "The CT gantry will not rotate after startup. Which components should I inspect?",
    "MRI images contain zipper artifacts. What usually causes this type of artifact?",
    "The ultrasound image is completely black even though the machine powers on. What are the possible causes?",
    "What preventive maintenance should be performed on a CT scanner to reduce unexpected failures?",
    "How can I differentiate between a detector failure and a detector calibration problem in a CT scanner?",
    "My imaging system repeatedly restarts during operation without displaying any error code. What are the most likely causes?",
]

for i, question in enumerate(sample_questions):
    print(f"\n--- Question {i+1} ---")
    print(f"Q: {question}")

    # Generate answer from fine-tuned model
    ft_answer = generate_answer(stage1_merged_model, stage1_merged_tokenizer, instruction=question)
    print(f"Fine-tuned Model A: {ft_answer}")

clear_gpu_memory()


--- Question 1 ---
Q: My CT scanner is producing circular ring artifacts on every scan. What is the most likely cause and what should I inspect first?
Fine-tuned Model A: This is a symptom of artifacts from the scintillation generator. In my CT scanner, I have a generator that generates scintillating material as part of the signal. This scintillating material is then transmitted to an attenuator that is used in the image reconstruction process. The scintillating material can be generated by either a cobalt chloride or a gadolinium chloride. The attenuators are either plastic or ceramic. It is quite common for these materials to generate an artifact which is a "ring" around the image region. This artifact is often associated with the presence of a tumor. It is not unusual for a tum

--- Question 2 ---
Q: The CT scanner stopped scanning because of a tube overheating alarm. What components are likely responsible?
Fine-tuned Model A: The CT scanner stopped scanning because the tube overhe

## Interactive Comparison Sheet: Base Model vs. Fine-tuned Model

In [22]:
import ipywidgets as widgets
from IPython.display import display, HTML
import pandas as pd

def compare_answers(question_index):
    question = sample_questions[question_index]
    base_answer = generate_answer(base_model, base_tokenizer, instruction=question)
    ft_answer = generate_answer(stage1_merged_model, stage1_merged_tokenizer, instruction=question)

    print(f"\n--- Question: {question} ---\n")

    # Display base model answer
    print("\n--- Base Model Answer ---")
    print(base_answer)

    # Display fine-tuned model answer
    print("\n--- Fine-tuned Model (Stage 1) Answer ---")
    print(ft_answer)

    # Store for CSV export
    global comparison_df
    comparison_df = pd.DataFrame({
        'Question': [question],
        'Base Model Answer': [base_answer],
        'Fine-tuned Model Answer': [ft_answer]
    })

    clear_gpu_memory()

def export_to_csv():
    if 'comparison_df' in globals() and not comparison_df.empty:
        csv_filename = 'model_comparison_results.csv'
        comparison_df.to_csv(csv_filename, index=False)
        print(f"\nComparison results exported to {csv_filename}")
    else:
        print("\nNo comparison data to export. Please select a question first.")

# Create dropdown for questions
question_options = [(q, i) for i, q in enumerate(sample_questions)]
question_dropdown = widgets.Dropdown(
    options=question_options,
    description='Select Question:',
    disabled=False,
)

# Create output widgets to capture printed text
output_widget = widgets.Output()

def on_dropdown_change(change):
    with output_widget:
        output_widget.clear_output()
        compare_answers(change.new)

question_dropdown.observe(on_dropdown_change, names='value')

# Export button
export_button = widgets.Button(description="Export to CSV")
def on_export_button_click(b):
    with output_widget:
        export_to_csv()
export_button.on_click(on_export_button_click)

# Display widgets
display(question_dropdown, export_button, output_widget)

print("Select a question from the dropdown to compare the models' answers.")
print("Then click 'Export to CSV' to save the results.")


Dropdown(description='Select Question:', options=(('My CT scanner is producing circular ring artifacts on ever…

Button(description='Export to CSV', style=ButtonStyle())

Output()

Select a question from the dropdown to compare the models' answers.
Then click 'Export to CSV' to save the results.


## Full Comparison Table

In [23]:
comparison_data = []

for question in sample_questions:
    base_answer = generate_answer(base_model, base_tokenizer, instruction=question)
    ft_answer = generate_answer(stage1_merged_model, stage1_merged_tokenizer, instruction=question)
    comparison_data.append({
        'Question': question,
        'Base Model Answer': base_answer,
        'Fine-tuned Model Answer': ft_answer,
        'Problem/Reason': '' # Placeholder for manual input
    })

comparison_table_df = pd.DataFrame(comparison_data)
display(comparison_table_df)

clear_gpu_memory()

,Question,Base Model Answer,Fine-tuned Model Answer,Problem/Reason
0,My CT scanner is producing circular ring artif...,The most likely cause of these artifacts would...,CT scanners are designed to produce circular r...,
1,The CT scanner stopped scanning because of a t...,The tube is overheating and the CT scanner is ...,The tube overheated due to a loose tube connec...,
2,How does quantization reduce model size?,Quantization reduces the model's size and ener...,The quantized model is 3x smaller than the ful...,
3,My MRI system displays a low helium pressure a...,The MRI system is displaying an alarm because ...,A:\n\n*\n\n*The helium pressure was low enough...,
4,"The ultrasound probe is connected, but the sys...","To troubleshoot this issue, first check that t...",1) The ultrasound probe is connected properly....,
5,The CT gantry will not rotate after startup. W...,* CT gantry (on/off)\n* CT head\n* patient tab...,The CT gantry will not rotate after startup.\n...,
6,MRI images contain zipper artifacts. What usua...,The most common reason for this type of artifa...,The zipper artifacts are caused by the MRI sca...,
7,The ultrasound image is completely black even ...,Ultrasonography machines generate a lot of hea...,The image is black because the ultrasound devi...,
8,What preventive maintenance should be performe...,Provide a written schedule of preventive maint...,- Schedule preventative maintenance visits:\n\...,
9,How can I differentiate between a detector fai...,A scanner is a device that takes a series of X...,The response of a CT scanner is the ability to...,


Comparison and Analysis:

Base Model: The base model provides a more general, yet still relevant, troubleshooting guide. It directly suggests checking for a damaged or faulty tube and its connector, and advises on replacement or vendor contact. This is a common and practical first step for many hardware failures.

Fine-tuned Model (Stage 1): The fine-tuned model, having been exposed to the medical equipment PDF, provides a more specific and nuanced diagnosis. It focuses on the cooling of the tube as the root cause, mentioning coolant levels and ambient temperature. It even starts to list specific CT tube warming parameters, indicating that it has learned details from the training document. The answer is more technical and diagnostic in nature, reflecting the type of information likely present in the non-instruction data it was trained on.

Conclusion: The fine-tuned model demonstrates a clear improvement in understanding the context of medical equipment failures, moving beyond generic troubleshooting to more specific diagnostic reasoning related to the cooling system, which is a critical aspect for CT scanners mentioned in the training data. The base model's answer, while not incorrect, is less specialized.